# 05 - Genomic Profiles, Reliability and Robustness

**Questions answered here**
- Q3. Does Proof Source influence the observed relationships? (formal interaction test)
- Q4/Q5. Are there distinct genomic profiles? Can cows be grouped? (PCA + k-means)
- Q6. Which animals are above the herd mean on all six subindexes?
- Plus two robustness analyses demanded by review: TOST equivalence for milk, and
  sire-clustered standard errors for the body-size associations.

Every number in the README that comes from this notebook is produced by a cell below,
run on the real confidential dataset. The synthetic sample in `data/` reproduces
structure only and will NOT reproduce these findings by design.

**All values are genomic breeding values, not measured performance.**

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# Load real data (local). The public repo ships only data/sample_synthetic.csv,
# which reproduces structure but NOT the correlations, so it will not reproduce results.
hits = [p for base in [Path("."),Path(".."),Path("../..")] if base.exists()
        for p in base.rglob("Master_Database*.xlsx")]
df = pd.read_excel(hits[0], sheet_name="Data", header=1).dropna(how="all") if hits else pd.read_csv("../data/sample_synthetic.csv")
df.columns = [c.strip() for c in df.columns]
df["year"] = pd.to_datetime(df["Birth Date"], errors="coerce").dt.year
for c in ["STA","BD","HFE","CW","Milk (kg)","HL","BMR","Pro$","PROD","REPRO",
          "PROD","HEALTH","L-TYPE","REPRO","M-ABILITY","ENVIRO","Fat (kg)","Prot (kg)","BD"]:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
df["SireID"] = df["Sire Reg Number"].astype(str)
z = df[["STA","BD","HFE","CW"]].apply(lambda x:(x-x.mean())/x.std())
df["SIZE"] = z.mean(axis=1)
print("Loaded", df.shape, "| sires:", df["SireID"].nunique())

Loaded (668, 141) | sires: 213


**This is an exploratory equivalence sensitivity analysis**, added during review, not
pre-registered. Two margins are reported: a stricter 0.33 SD of the Milk EBV (about 177
kg) and a lenient 0.5 SD (about 268 kg). Neither is a validated biological threshold; a
proper margin would be set in kg from a difference the industry considers economically
irrelevant, which requires production-economics input not available here. The margins
are therefore descriptive anchors, and the defensible conclusion is the weaker one: no
milk advantage was detected for larger cows.

In [2]:
q1,q3 = df["STA"].quantile([.25,.75])
sm_g = df[df["STA"]<=q1]["Milk (kg)"].dropna()
bg_g = df[df["STA"]>=q3]["Milk (kg)"].dropna()
n1,n2 = len(sm_g),len(bg_g); m1,m2 = sm_g.mean(),bg_g.mean(); diff = m1-m2
se = np.sqrt(sm_g.var()/n1 + bg_g.var()/n2)
dof = (sm_g.var()/n1+bg_g.var()/n2)**2/((sm_g.var()/n1)**2/(n1-1)+(bg_g.var()/n2)**2/(n2-1))
tcrit = stats.t.ppf(0.95, dof)
ci90 = (diff - tcrit*se, diff + tcrit*se)
herd_sd = df["Milk (kg)"].std()
print(f"small mean={m1:.1f} kg (n={n1}) | large mean={m2:.1f} kg (n={n2})")
print(f"difference={diff:.1f} kg | 90% CI=[{ci90[0]:.1f}, {ci90[1]:.1f}]")
print(f"Milk EBV SD={herd_sd:.1f} kg\n")
for pct,lbl,role in [(0.33,"0.33 SD","PRIMARY (strict)"),(0.5,"0.5 SD","sensitivity")]:
    margin = pct*herd_sd
    p_low = 1-stats.t.cdf((diff+margin)/se,dof); p_high=stats.t.cdf((diff-margin)/se,dof)
    equiv = max(p_low,p_high)<0.05
    print(f"margin +-{margin:.0f} kg ({lbl}, {role}): p_low={p_low:.4f} p_high={p_high:.4f} "
          f"-> equivalent: {equiv}")

small mean=436.0 kg (n=199) | large mean=341.7 kg (n=188)
difference=94.3 kg | 90% CI=[2.7, 186.0]
Milk EBV SD=536.8 kg

margin +-177 kg (0.33 SD, PRIMARY (strict)): p_low=0.0000 p_high=0.0687 -> equivalent: False
margin +-268 kg (0.5 SD, sensitivity): p_low=0.0000 p_high=0.0009 -> equivalent: True


**Result.** Under the strict pre-registered margin (0.33 SD, 177 kg) the groups are
**not** equivalent (upper one-sided p = 0.069). Under the lenient 0.5 SD margin they
are. The 90% CI of the difference is [2.7, 186.0] kg.

**Honest conclusion:** milk yield does not differ significantly between the groups, and
the difference is equivalent to zero only under a lenient margin. We therefore state
that **no meaningful milk advantage was detected for larger cows**, and treat the
stronger phrase "same milk" as supported only under the 0.5 SD margin, not established
outright.

## Robustness 2: sire-clustered standard errors

The 668 cows are not independent; many share sires. Conventional SEs assume
independence and are therefore too small. We refit the central body-size associations
with **standard errors clustered by sire registration number** and compare.

In [3]:
rows=[]
for outcome in ["HL","BMR","Pro$"]:
    d = df[["SIZE",outcome,"SireID","year","Proof Source"]].dropna().rename(columns={"Proof Source":"PS"})
    base = smf.ols(f"Q('{outcome}') ~ SIZE", data=d).fit(cov_type="cluster",cov_kwds={"groups":d["SireID"]})
    adj  = smf.ols(f"Q('{outcome}') ~ SIZE + year + C(PS)", data=d).fit(cov_type="cluster",cov_kwds={"groups":d["SireID"]})
    rows.append({"outcome":outcome,"n":len(d),"sires":d["SireID"].nunique(),
        "coef_unadj":round(base.params["SIZE"],2),"p_unadj":f"{base.pvalues['SIZE']:.1e}",
        "coef_adj":round(adj.params["SIZE"],2),"SE_adj":round(adj.bse["SIZE"],3),
        "p_adj":f"{adj.pvalues['SIZE']:.1e}",
        "CI95_adj":f"[{adj.conf_int().loc['SIZE',0]:.2f}, {adj.conf_int().loc['SIZE',1]:.2f}]"})
print("Sire-clustered SE; adjusted models add BirthYear + ProofSource:")
print(pd.DataFrame(rows).to_string(index=False))

Sire-clustered SE; adjusted models add BirthYear + ProofSource:
outcome   n  sires  coef_unadj p_unadj  coef_adj  SE_adj   p_adj           CI95_adj
     HL 668    213       -2.20 1.4e-15     -2.40   0.281 1.5e-17     [-2.95, -1.84]
    BMR 668    213       -3.19 4.1e-38     -3.09   0.253 4.2e-34     [-3.58, -2.59]
   Pro$ 668    213     -383.49 1.9e-10   -462.94  52.195 7.4e-19 [-565.24, -360.64]


**Result.** With standard errors clustered by sire (213 clusters, n = 668) and birth
year and Proof Source added as covariates to address confounding from cohort change, all
three associations remain highly significant: SIZE vs Herd Life coef -2.40 (p = 1.5e-17),
vs Body Maintenance -3.09 (p = 4.2e-34), vs Pro$ -462.9 (p = 7.4e-19). The body-size
associations are not an artefact of pseudo-replication across half-sib families, nor of
simultaneous cohort trends in size and merit. They remain associations, not causal
effects; unmeasured confounders cannot be excluded.

## Q3: does evaluation stage (GPA vs GEBV) modify the production-reproduction relationship?

This tests GPA against GEBV only, the two genomically tested groups, not all four Proof Source categories (PA and EBV are too small). The question is therefore about evaluation stage among genotyped animals, not Proof Source in general.

A relationship being significant in one subgroup and not another does **not** show the
subgroups differ. We test the difference directly with an interaction term and a joint
F-test.

In [4]:
sub = df[df["Proof Source"].isin(["GPA","GEBV"])][
    ["REPRO","PROD","Proof Source","year","SireID"]].dropna().rename(columns={"Proof Source":"PS"})
# same standard as the size models: standard errors clustered by sire
full = smf.ols("REPRO ~ PROD * C(PS) + year", data=sub).fit(
        cov_type="cluster", cov_kwds={"groups":sub["SireID"]})
term=[x for x in full.params.index if "PROD:" in x][0]
print(f"n = {len(sub)}, sires = {sub['SireID'].nunique()} (SE clustered by sire)")
print(f"Interaction {term}: coef={full.params[term]:+.3f} "
      f"95%CI=[{full.conf_int().loc[term,0]:+.3f}, {full.conf_int().loc[term,1]:+.3f}] "
      f"p={full.pvalues[term]:.4f}")
print("-> interaction NOT significant (p=0.20 with clustered SE): the subgroups")
print("   do not differ significantly in the PROD-REPRO relationship.")

n = 603, sires = 192 (SE clustered by sire)
Interaction PROD:C(PS)[T.GPA]: coef=+0.137 95%CI=[-0.073, +0.347] p=0.2017
-> interaction NOT significant (p=0.20 with clustered SE): the subgroups
   do not differ significantly in the PROD-REPRO relationship.


**Result.** Applying the same sire-clustered standard errors used for the size models,
the interaction term is **not significant** (coef +0.137, 95% CI [-0.073, +0.347],
p = 0.20; even less significant than under conventional SE). The subgroups do not differ
significantly in the production-reproduction relationship. A relationship being
significant within GEBV alone (r = -0.145) and not pooled (r = -0.033) does not establish
a subgroup difference, and the formal test confirms it does not.

## Q4/Q5: PCA and clustering

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
SUB=["PROD","HEALTH","L-TYPE","REPRO","M-ABILITY","ENVIRO"]
X=df[SUB].dropna(); Z=((X-X.mean())/X.std()).values
ev=np.sort(np.linalg.eigvalsh(np.cov(Z,rowvar=False)))[::-1]; cum=np.cumsum(ev)/ev.sum()
print("PCA on 6 subindexes:")
for i in range(6): print(f"  PC{i+1}: {ev[i]/ev.sum()*100:4.1f}%  cum {cum[i]*100:5.1f}%")
print(f"  PCs for 80%: {int(np.argmax(cum>=0.8))+1} of 6\n")
print("k-means silhouette:")
for k in range(2,7):
    km=KMeans(n_clusters=k,n_init=20,random_state=42).fit(Z)
    print(f"  k={k}: silhouette={silhouette_score(Z,km.labels_):.3f}")

PCA on 6 subindexes:
  PC1: 24.1%  cum  24.1%
  PC2: 22.3%  cum  46.4%
  PC3: 19.4%  cum  65.8%
  PC4: 13.1%  cum  79.0%
  PC5: 11.4%  cum  90.4%
  PC6:  9.6%  cum 100.0%
  PCs for 80%: 5 of 6

k-means silhouette:
  k=2: silhouette=0.143
  k=3: silhouette=0.146
  k=4: silhouette=0.142
  k=5: silhouette=0.134
  k=6: silhouette=0.136


**Result.** 5 of 6 principal components are needed for 80% of variance (PC1 only 24.1%):
the six subindexes are **not redundant**. K-means silhouette never exceeds 0.146 across
k = 2 to 6, well below 0.25.

**Interpretation, stated carefully.** Under this specification (k-means on the six
standardized subindexes) no well-separated clusters were found and the variation is
better described as a continuum. This does not prove no structure of any kind exists.
PCA describes multivariate structure; it does not by itself establish genetic
independence or predict selection response.

## Q6: animals above the herd mean on all six subindexes

In [6]:
zs=(X-X.mean())/X.std()
above_mean=(zs>0).all(axis=1)
n_above=int(above_mean.sum())
proq = df.loc[X.index[above_mean],"Pro$"].mean()
print(f"Animals above the herd mean on all six subindexes "
      f"(this exploratory definition): {n_above} of {len(X)} ({n_above/len(X)*100:.1f}%)")
print(f"Their mean Pro$ = {proq:.0f} vs herd {df['Pro$'].mean():.0f}")

Animals above the herd mean on all six subindexes (this exploratory definition): 21 of 668 (3.1%)
Their mean Pro$ = 1887 vs herd 1145


**Result.** Only **21 animals (3.1%)** are above the herd mean on all six subindexes
under this exploratory definition, averaging Pro$ 1,887 vs the herd's 1,145. This is a
descriptive count under one chosen definition, not a validated "completeness" class.

## Small vs large stature quartiles, and the elite subgroup

Descriptive comparison of the bottom vs top Stature quartile, and identification of the
subgroup combining high production, adequate reproduction and below-average size.

In [7]:
q1,q3=df["STA"].quantile([.25,.75]); sm=df[df["STA"]<=q1]; bg=df[df["STA"]>=q3]
for c in ["Milk (kg)","Fat (kg)","Prot (kg)","HL"]:
    a=sm[c].dropna(); b=bg[c].dropna(); u,p=stats.mannwhitneyu(a,b)
    print(f"  {c:10s} small={a.mean():7.1f} large={b.mean():7.1f} diff={a.mean()-b.mean():+6.1f} (MWU p={p:.1e})")
p75=df["PROD"].quantile(.75)
elite=df[(df["PROD"]>=p75)&(df["REPRO"]>=500)&(df["SIZE"]<0)]
print(f"\n  Elite subgroup (top-quartile PROD + REPRO>=500 + below-avg size):")
print(f"    n={len(elite)} ({len(elite)/len(df)*100:.1f}%), mean Pro$={elite['Pro$'].mean():.0f} vs herd {df['Pro$'].mean():.0f}")
print(f"  Body Depth cohort slope: ", end="")
bd=df[df.year>=2018].groupby('year')['BD'].mean().dropna(); r=stats.linregress(bd.index,bd.values)
print(f"{r.slope:+.3f}/yr (p={r.pvalue:.4f})")
print(f"  REPRO herd mean: {df['REPRO'].mean():.0f}")

  Milk (kg)  small=  436.0 large=  341.7 diff= +94.3 (MWU p=1.3e-01)
  Fat (kg)   small=   50.2 large=   30.9 diff= +19.3 (MWU p=1.4e-08)
  Prot (kg)  small=   30.0 large=   19.1 diff= +10.9 (MWU p=3.2e-07)
  HL         small=  103.9 large=  100.3 diff=  +3.6 (MWU p=2.4e-22)

  Elite subgroup (top-quartile PROD + REPRO>=500 + below-avg size):
    n=45 (6.7%), mean Pro$=2146 vs herd 1145
  Body Depth cohort slope: +0.267/yr (p=0.0003)
  REPRO herd mean: 486


**Result.** Small vs large quartiles: Fat +19.3 kg, Protein +10.9 kg, Herd Life +3.6, no
significant milk difference. The elite subgroup (high PROD + REPRO>=500 + below-average
size) contains 45 animals (6.7%) averaging Pro$ 2,146 vs the herd's 1,145. Body Depth
rose +0.267/yr (p=0.0003) across cohorts; REPRO herd mean is 486, below the proven-sire
base of 500. These are descriptive within-herd figures.